# Vehicle Maintenance - ML Model Training
This notebook contains the complete pipeline to train and evaluate a Random Forest Classifier on component-level maintenance rules.

### **Cell 1: Imports and Configuration**

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Configuration
INPUT_FILE = 'vehicle_data.csv'
MODEL_FILE = 'vehicle_model.pkl'
CLASS_NAMES = ['Healthy', 'Warning', 'Critical']

### **Cell 2: Load and Prepare Data**

In [2]:
# Load dataset
if not os.path.exists(INPUT_FILE):
    print(f"Error: {INPUT_FILE} NOT found. Please upload it to your workspace.")
else:
    df = pd.read_csv(INPUT_FILE)
    
    # Preprocessing
    X = df.drop('health_status', axis=1)
    y = df['health_status']
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Data loaded! Training samples: {len(X_train)} | Testing samples: {len(X_test)}")

Data loaded! Training samples: 8000 | Testing samples: 2000


### **Cell 3: Train the Model with Pipeline**

In [3]:
print("Training Random Forest Classifier Pipeline...")

# Define preprocessing for categorical columns
categorical_features = ['component']
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Keep numeric features as they are
numeric_features = ['current_km', 'current_months', 'condition_metric_value']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features),
        ('num', 'passthrough', numeric_features)
    ])
    
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

model.fit(X_train, y_train)

# Generate Predictions for the next cells
y_pred = model.predict(X_test)
print("Model training complete!")

Training Random Forest Classifier Pipeline...
Model training complete!


### **Cell 4: Overall Accuracy & Classification Report**

In [4]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {accuracy * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

Test accuracy: 99.65%

Classification Report:
              precision    recall  f1-score   support

     Healthy       1.00      0.99      1.00       789
     Warning       1.00      1.00      1.00       403
    Critical       0.99      1.00      1.00       808

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



### **Cell 5: Confusion Matrix Visualization**

In [5]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='coolwarm', 
            xticklabels=CLASS_NAMES, 
            yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix', fontsize=16)
plt.xlabel('Predicted Labels', fontsize=12)
plt.ylabel('True Labels', fontsize=12)
plt.tight_layout()
plt.show()

#### **Saved Confusion Matrix Plot**
![Confusion Matrix](confusion_matrix.png)

### **Cell 6: Learning Curve (Generalization & Data Sufficiency)**

In [6]:
from sklearn.model_selection import learning_curve

print("Generating Learning Curve...")
train_sizes, train_scores, test_scores = learning_curve(
    model, X, y, cv=5, n_jobs=-1, 
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy',
    random_state=42
)
train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

plt.figure(figsize=(10, 6))
plt.title("Learning Curve (Random Forest Classifier)", fontsize=14)
plt.xlabel("Training Examples", fontsize=12)
plt.ylabel("Accuracy Score", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                 train_scores_mean + train_scores_std, alpha=0.15, color="crimson")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                 test_scores_mean + test_scores_std, alpha=0.15, color="teal")
plt.plot(train_sizes, train_scores_mean, 'o-', color="crimson", label="Training Score", linewidth=2)
plt.plot(train_sizes, test_scores_mean, 'o-', color="teal", label="Cross-Validation Score", linewidth=2)
plt.legend(loc="best", fontsize=11)
plt.tight_layout()
plt.show()

Generating Learning Curve...


#### **Saved Learning Curve Plot**
![Learning Curve](learning_curve.png)

### **Cell 7: Model Loss vs Number of Trees (Estimators)**

In [7]:
from sklearn.metrics import log_loss

print("Generating Loss vs Number of Trees...")
n_estimators_range = list(range(5, 155, 5))
train_losses = []
test_losses = []

for n_est in n_estimators_range:
    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=n_est, random_state=42, n_jobs=-1))
    ])
    clf.fit(X_train, y_train)
    
    train_probs = clf.predict_proba(X_train)
    test_probs = clf.predict_proba(X_test)
    
    train_losses.append(log_loss(y_train, train_probs))
    test_losses.append(log_loss(y_test, test_probs))

plt.figure(figsize=(10, 6))
plt.title("Model Loss vs Number of Trees (Estimators)", fontsize=14)
plt.xlabel("Number of Trees", fontsize=12)
plt.ylabel("Log Loss (Cross-Entropy Loss)", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.plot(n_estimators_range, train_losses, 's-', color="crimson", label="Training Loss", linewidth=2)
plt.plot(n_estimators_range, test_losses, 'o-', color="teal", label="Test Loss", linewidth=2)
plt.legend(loc="best", fontsize=11)
plt.tight_layout()
plt.show()

Generating Loss vs Number of Trees...


#### **Saved Loss vs Trees Plot**
![Model Loss vs Number of Trees](loss_vs_trees.png)

### **Cell 8: Save Model**

In [8]:
joblib.dump(model, MODEL_FILE)
print(f"Model saved successfully as '{MODEL_FILE}'.")

Model saved successfully as 'vehicle_model.pkl'.
